In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

path = 'data/ichccaesuppall.xlsx'
raw = pd.read_excel(path, sheet_name="Table 1.1", header=None)

hdr = None
for i in range(min(80, len(raw))):
    row = raw.iloc[i].astype(str).tolist()
    if any("country" in x.lower() for x in row) and any(re.search(r"\b1996\b", x) for x in row):
        hdr = i
        break
if hdr is None:
    raise ValueError("Header row not found")

df = raw.iloc[hdr+1:].copy()
df.columns = raw.iloc[hdr].tolist()
df = df.rename(columns={df.columns[0]: "Country"})

year_map = {}
for c in df.columns:
    m = re.search(r"\b(19\d{2}|20\d{2})\b", str(c))
    if m:
        year_map[c] = int(m.group(1))

keep = ["Country"] + list(year_map.keys())
df = df[keep]
df = df.rename(columns=year_map)

for y in year_map.values():
    df[y] = pd.to_numeric(df[y], errors="coerce")

df["Country"] = df["Country"].astype(str).str.strip()
df = df[df["Country"].ne("")]

countries = ["United States", "Mexico", "Brazil", "Poland", "Czech Republic"]
years = sorted([c for c in df.columns if isinstance(c, int)])

plot_df = df[df["Country"].isin(countries)].set_index("Country")[years].T

plt.figure()
for c in countries:
    if c in plot_df.columns:
        s = plot_df[c].dropna()
        plt.plot(s.index, s.values, label=c)
plt.xlabel("Year")
plt.ylabel("Compensation Index (US = 100)")
plt.title("Hourly Compensation Trends by Country (1996–2012)")
plt.legend()
plt.show()

latest_year = max(years)
latest = plot_df.loc[latest_year].dropna()

plt.figure()
plt.bar(latest.index, latest.values)
plt.ylabel("Compensation Index (US = 100)")
plt.title(f"Labor Cost Comparison ({latest_year})")
plt.show()
